In [1]:
import pandas as pd 
import numpy as np

In [2]:
df=pd.read_csv("banana_quality_dataset.csv")

In [3]:
#basic checks
df.head()

,sample_id,variety,region,quality_score,quality_category,ripeness_index,ripeness_category,sugar_content_brix,firmness_kgf,length_cm,weight_g,harvest_date,tree_age_years,altitude_m,rainfall_mm,soil_nitrogen_ppm
0,1,Manzano,Colombia,1.88,Processing,2.11,Turning,16.83,3.53,21.44,146.92,2023-10-16,13.7,58.2,2440.5,183.6
1,2,Plantain,Guatemala,2.42,Processing,4.25,Ripe,16.73,4.09,26.11,160.48,2023-10-14,5.1,280.2,2374.6,109.8
2,3,Burro,Ecuador,3.57,Premium,6.24,Overripe,21.34,1.63,25.20,225.27,2023-09-08,17.7,1246.9,1191.5,147.7
3,4,Manzano,Ecuador,2.21,Processing,5.39,Ripe,16.75,3.31,13.08,137.80,2023-10-07,13.0,1150.2,2845.1,92.8
4,5,Red Dacca,Ecuador,2.35,Processing,5.84,Ripe,16.90,3.07,12.98,227.84,2023-10-02,4.8,526.0,2136.9,129.7


In [4]:
df.drop(["sample_id","harvest_date"],axis=1,inplace=True)  

In [5]:
#target variable quality category
y=df['quality_score']

In [6]:
x =df.drop(['quality_score'],axis=1)

In [7]:
x.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 13 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   variety             1000 non-null   object 
 1   region              1000 non-null   object 
 2   quality_category    1000 non-null   object 
 3   ripeness_index      1000 non-null   float64
 4   ripeness_category   1000 non-null   object 
 5   sugar_content_brix  1000 non-null   float64
 6   firmness_kgf        1000 non-null   float64
 7   length_cm           1000 non-null   float64
 8   weight_g            1000 non-null   float64
 9   tree_age_years      1000 non-null   float64
 10  altitude_m          1000 non-null   float64
 11  rainfall_mm         1000 non-null   float64
 12  soil_nitrogen_ppm   1000 non-null   float64
dtypes: float64(9), object(4)
memory usage: 101.7+ KB


In [8]:
#apply columntransformer 
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OrdinalEncoder,StandardScaler
ct=ColumnTransformer(transformers=[("oe",OrdinalEncoder(),[0,1,2,4]),
                                   ("sc",StandardScaler(),[3,5,6,7,8,9,10,11,12])])

In [9]:
x_transformed=ct.fit_transform(x)

In [10]:
x_transformed=pd.DataFrame(x_transformed,columns=x.columns)
x_transformed

,variety,region,quality_category,ripeness_index,ripeness_category,sugar_content_brix,firmness_kgf,length_cm,weight_g,tree_age_years,altitude_m,rainfall_mm,soil_nitrogen_ppm
0,5.0,1.0,2.0,3.0,-1.102626,-0.829400,0.638798,0.271721,-0.362627,0.541038,-1.557632,0.828641,1.478299
1,6.0,4.0,2.0,2.0,0.118602,-0.878570,1.073376,1.086240,-0.086672,-1.108701,-1.037919,0.711835,0.100944
2,1.0,3.0,1.0,1.0,1.254230,1.388129,-0.835662,0.927522,1.231846,1.308358,1.225172,-1.385170,0.808285
3,5.0,3.0,2.0,2.0,0.769163,-0.868736,0.468071,-1.186389,-0.548225,0.406757,0.998792,1.545781,-0.216332
4,7.0,3.0,2.0,2.0,1.025963,-0.794982,0.281824,-1.203830,1.284148,-1.166250,-0.462490,0.290520,0.472345
...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,1.0,3.0,0.0,2.0,0.512362,1.250456,-0.168275,1.407163,0.643306,0.291659,-0.734754,0.100866,1.280467
996,2.0,7.0,2.0,1.0,1.539564,-1.188335,-0.548530,-1.306735,0.016506,0.598587,-0.958090,-1.138442,1.420442
997,6.0,3.0,2.0,0.0,-1.502094,-0.662225,-0.331241,-0.579424,0.922720,1.442639,-0.094945,1.741284,1.595878
998,3.0,4.0,2.0,0.0,-1.542040,-0.731062,0.134378,1.121123,-0.045564,-0.514028,1.496265,-1.342099,-0.352575


In [11]:
#feature selection
from sklearn.feature_selection import mutual_info_regression
mf=mutual_info_regression(x_transformed,y)
mf=pd.Series(mf)
mf.index=x_transformed.columns
mf.sort_values(ascending=False)

quality_category      0.899355
ripeness_category     0.326210
sugar_content_brix    0.270958
ripeness_index        0.248317
length_cm             0.137056
firmness_kgf          0.038136
weight_g              0.029296
rainfall_mm           0.027548
altitude_m            0.027305
soil_nitrogen_ppm     0.019739
variety               0.004235
region                0.000000
tree_age_years        0.000000
dtype: float64

In [12]:
x.drop(['tree_age_years','variety','region','soil_nitrogen_ppm','altitude_m'],axis=1,inplace=True)

In [13]:
from sklearn.model_selection import train_test_split
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2)

In [14]:
ct=ColumnTransformer(transformers=[("oe",OrdinalEncoder(),[0,2]),
                                  ("sc",StandardScaler(),[1,3,4,5,6,7])])

In [15]:
x_train_tran=ct.fit_transform(x_train)
x_train_tran=pd.DataFrame(x_train_tran)

In [16]:
x_test_tran=ct.transform(x_test)
x_test_tran=pd.DataFrame(x_test_tran)

In [19]:
from sklearn.model_selection import cross_val_score
score=cross_val_score(kn,x_transformed,y,cv=5)
score

array([0.7835618 , 0.81872917, 0.81938143, 0.80977705, 0.82323384])

In [18]:
from sklearn.neighbors import KNeighborsRegressor
kn=KNeighborsRegressor(n_neighbors=5)
kn.fit(x_train_tran,y_train)

KNeighborsRegressor()

In [20]:
#predct
y_pred=kn.predict(x_test_tran)

In [21]:
#evolution
from sklearn.metrics import r2_score
r2=r2_score(y_test,y_pred)
r2

0.9319061136921477

In [22]:
y_train_pred=kn.predict(x_train_tran)

In [23]:
r=r2_score(y_train,y_train_pred)
r

0.9535383562780483

# Hyper parameter tuning 

In [25]:
import warnings 
warnings.filterwarnings("ignore")

In [30]:
from sklearn.model_selection import GridSearchCV
from sklearn.neighbors import KNeighborsRegressor
kn=KNeighborsRegressor()

param_grid={
    "n_neighbors":[3,5,7,9],
    "weights":['uniform','distance'],
    'p':[1,2]
    
}
grid=GridSearchCV(estimator=kn,param_grid=param_grid,cv=5,scoring='r2',verbose=1)
grid.fit(x_train_tran,y_train)

Fitting 5 folds for each of 16 candidates, totalling 80 fits


GridSearchCV(cv=5, estimator=KNeighborsRegressor(),
             param_grid={'n_neighbors': [3, 5, 7, 9], 'p': [1, 2],
                         'weights': ['uniform', 'distance']},
             scoring='r2', verbose=1)

In [31]:
print("Best parameter:",grid.best_params_)
print("best CV Accuracy:",grid.best_score_)

Best parameter: {'n_neighbors': 5, 'p': 1, 'weights': 'distance'}
best CV Accuracy: 0.9230076216667928


In [ ]:
param_grid={
    "n_neighbors":[3,5,7,9],
    "weights":['uniform','distance'],
    'p':[1,2]
    
}

In [36]:
from sklearn.model_selection import RandomizedSearchCV
random_search=RandomizedSearchCV(
    kn,
    param_distributions=param_grid,
    n_iter=30,
    cv=5,
    scoring='r2',
    verbose=1,n_jobs=-1)
random_search.fit(x_train_tran,y_train)
 

Fitting 5 folds for each of 16 candidates, totalling 80 fits


RandomizedSearchCV(cv=5, estimator=KNeighborsRegressor(), n_iter=30, n_jobs=-1,
                   param_distributions={'n_neighbors': [3, 5, 7, 9],
                                        'p': [1, 2],
                                        'weights': ['uniform', 'distance']},
                   scoring='r2', verbose=1)

In [37]:
print("Best parameters:",random_search.best_params_)
print("Best CV Accuracy:",random_search.best_score_)

Best parameters: {'weights': 'distance', 'p': 1, 'n_neighbors': 5}
Best CV Accuracy: 0.9230076216667928


In [38]:
from sklearn.model_selection import cross_val_score
score=cross_val_score(kn,x_train_tran,y_train,cv=5)
score

array([0.92140008, 0.9037075 , 0.92129538, 0.9276948 , 0.91549464])

In [39]:
score.mean()

np.float64(0.9179184805238643)